In [1]:
import pickle
import re
import json 
import ast
import json
with open("with-durations-0shot-preds.pkl", "rb") as f:
    without_durations = pickle.load(f)

In [2]:
without_durations

[{'question': 'Of [Aaron Brooks member of sports team Sacramento Kings] and [Choi Sung-yong number of matches played/races/starts +64], which one was shorter?',
  'gold_entities': {'Aaron Brooks member of sports team Sacramento Kings'},
  'pred_answer': '<|start_header_id|>assistant<|end_header_id|>\n\n{"answer_triple": Aaron Brooks member of sports team Sacramento Kings from 2012 to 2013. (Duration: 1 years)}<|eot_id|>',
  'accuracy': 1},
 {'question': 'Of [Choi Sung-yong number of matches played/races/starts +64] and [Aaron Brooks member of sports team Sacramento Kings], which one was shorter?',
  'gold_entities': {'Aaron Brooks member of sports team Sacramento Kings'},
  'pred_answer': '<|start_header_id|>assistant<|end_header_id|>\n\n{"answer_triple": "Aaron Brooks member of sports team Sacramento Kings from 2012 to 2013. (Duration: 1 years)"}<|eot_id|>',
  'accuracy': 1},
 {'question': 'Which event lasted for a shorter time, [Sidney Verba of American Political Science Association]

In [3]:
invalid_counter = 0
correct = 0
for sample in without_durations:
    
    if sample["accuracy"]==1:
        correct += 1
        matches = re.findall(r"\{(.*?)\}", sample["pred_answer"])
        if len(matches)==0:
            invalid_counter +=1
            continue
        cleaned_text = re.sub(re.escape(str(sample["gold_entities"])[2:-2]), "", matches[0], flags=re.IGNORECASE)
        cleaned_text= re.sub(r'"answer_triple"\s*:\s*|\bfrom\b|\bto\b|[\'".\[\],]','',cleaned_text)
        cleaned_text = re.sub(r'\b\d+\b', '', cleaned_text)
        cleaned_text = re.sub(r'\s*\(Duration:\s*(?:\d+\s*)?years?\)\s*', ' ', cleaned_text)
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
        
        if cleaned_text.replace(" ", "") == "":
            continue
        #print(str(sample["gold_entities"]))
        print(sample["pred_answer"])
        print(cleaned_text)
        print("###########")
        invalid_counter +=1

<|start_header_id|>assistant<|end_header_id|>

{"answer_triple": Anna Feodorovna (Princess Juliane of Saxe-Coburg and Gotha) end cause annulment from 1796 to 1820. (Duration: 24 years) or Hank Brown electoral district Colorado from 1991 to 1993. (Duration: 2 years)}<|eot_id|>
Anna Feodorovna (Princess Juliane of Saxe-Coburg and Gotha) end cause annulment or
###########
<|start_header_id|>assistant<|end_header_id|>

{"answer_triple": "Sigismund replaces Jobst of Moravia from 1411 to 1437. (Duration: 26 years)" if int("26") > int("1") else "Isadora Duncan end cause divorce from 1922 to 1923. (Duration: 1 years)" }<|eot_id|>
if int() > int() else Isadora Duncan end cause divorce
###########
<|start_header_id|>assistant<|end_header_id|>

{"answer_triple": "David II of Scotland replaced by Robert II of Scotland from 1329 to 1371. (Duration: 42 years)" or "German Confederation contains the administrative territorial entity Hohenzollern-Sigmaringen from 1815 to 1866. (Duration: 51 years)"}<|e

In [4]:
correct

1689

In [5]:
invalid_counter

245

In [6]:
(correct-invalid_counter)/2000

0.722

In [7]:
template_invalid_counter = 0
template_correct = 0
for i,sample in enumerate(without_durations):
    if i%2==1:
        if sample["accuracy"]==1:
            template_correct += 1
            matches = re.findall(r"\{(.*?)\}", sample["pred_answer"])
            if len(matches)==0:
                template_invalid_counter +=1
                continue
            cleaned_text = re.sub(re.escape(str(sample["gold_entities"])[2:-2]), "", matches[0], flags=re.IGNORECASE)
            cleaned_text = re.sub(r'"answer_triple"\s*:\s*|\bfrom\b|\bto\b|[\'".\[\],]','',cleaned_text)
            cleaned_text = re.sub(r'\b\d+\b', '', cleaned_text)
            cleaned_text = re.sub(r'\s*\(Duration:\s*(?:\d+\s*)?years?\)\s*', ' ', cleaned_text)
            cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

            if cleaned_text.replace(" ", "") == "":
                continue
            print(str(sample["gold_entities"]))
            print(sample["pred_answer"])
            print(cleaned_text)
            print("###########")
            template_invalid_counter +=1

{'Hank Brown electoral district Colorado'}
<|start_header_id|>assistant<|end_header_id|>

{"answer_triple": Anna Feodorovna (Princess Juliane of Saxe-Coburg and Gotha) end cause annulment from 1796 to 1820. (Duration: 24 years) or Hank Brown electoral district Colorado from 1991 to 1993. (Duration: 2 years)}<|eot_id|>
Anna Feodorovna (Princess Juliane of Saxe-Coburg and Gotha) end cause annulment or
###########
{'German Confederation contains the administrative territorial entity Hohenzollern-Sigmaringen'}
<|start_header_id|>assistant<|end_header_id|>

{"answer_triple": "David II of Scotland replaced by Robert II of Scotland from 1329 to 1371. (Duration: 42 years)" or "German Confederation contains the administrative territorial entity Hohenzollern-Sigmaringen from 1815 to 1866. (Duration: 51 years)"}<|eot_id|>
David II of Scotland replaced by Robert II of Scotland or
###########
{'Gerardo Torrado number of points/goals/set scored +6'}
<|start_header_id|>assistant<|end_header_id|>

{"a

In [8]:
(template_correct-template_invalid_counter)/1000

0.588